# Arc2Face — Main Model Generation on LFW  ⭐ RUN THIS NOTEBOOK FIRST

**Course Project — Image Processing**
**Federal University of Maranhão (UFMA) — CCET**

Authors: Louise Reis Mendes, Raianny Cristina Ferreira da Silva, Yasmin Cantanhede Santos, Haroldo Gomes

---

## ⚠️ Execution order (important)

This is a reproduction of **Arc2Face** (Paraperas Papantoniou et al., ECCV 2024), the **main model**
of our study. The full project compares three models on the same identities:

```
1. THIS NOTEBOOK  →  Arc2Face          (main model)      ← run first
2. 02_ipadapter_faceid_plusv2.ipynb  →  IP-Adapter-FaceID-PlusV2  (baseline 1)
3. 03_photomaker.ipynb               →  PhotoMaker                (baseline 2)
4. 04_evaluation.ipynb               →  Metrics (ID Sim, LPIPS, FID)
```

**Why first?** This notebook is the one that **selects and freezes the 50 evaluation
identities** and writes the list to `arc2face_processed.json` on Google Drive. The baseline
notebooks then read that exact list, so all three models are evaluated on **identical inputs**.
If you run a baseline before this notebook, the identity list will not exist yet.

## What Arc2Face does

Arc2Face conditions a Stable Diffusion v1.5 backbone **exclusively on an ArcFace identity
embedding** (no text describing the person). A frozen pseudo-prompt of the form
`"photo of a person <id>"` has its `<id>` token replaced by the projected ArcFace vector,
which then drives the UNet cross-attention. This is what gives Arc2Face its strong identity
preservation compared to text+identity methods.

> **Hardware:** NVIDIA A100 (or T4) GPU. Runtime → Change runtime type → GPU.

## 1. Dependency Installation

Pinned versions for reproducibility. `transformers` and `peft` are installed with `--no-deps`
because Arc2Face requires `transformers==4.36.0`, which the default Colab environment would
otherwise upgrade and break.

In [ ]:
!pip install -q diffusers==0.29.2
!pip install -q transformers==4.36.0 --no-deps
!pip install -q tokenizers==0.15.2 safetensors==0.4.1 huggingface_hub
!pip install -q peft==0.9.0 --no-deps
!pip install -q accelerate einops
!pip install -q insightface onnxruntime-gpu
!pip install -q lpips

import pkg_resources
for pkg in ['diffusers', 'transformers', 'peft', 'insightface']:
    v = pkg_resources.get_distribution(pkg).version
    print(f'{pkg}: {v}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
accelerate 1.14.0 requires safetensors>=0.4.3, but you have safetensors 0.4.1 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━

/tmp/ipykernel_515/3945125450.py:9: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


## 2. Clone Repository and Download Model Weights

We clone the official Arc2Face repository (for the `CLIPTextModelWrapper` and
`project_face_embs` utilities) and download the pretrained weights from the Hugging Face hub:
the adapted CLIP text encoder, the fine-tuned UNet, and the ArcFace recognition model.

In [ ]:
import os
from huggingface_hub import hf_hub_download

# Clone the official Arc2Face repository and enter it
!git clone https://github.com/foivospar/Arc2Face.git
%cd Arc2Face

os.makedirs('models/antelopev2', exist_ok=True)

print('Downloading Arc2Face encoder, UNet and ArcFace weights...')
hf_hub_download(repo_id='FoivosPar/Arc2Face', filename='arc2face/config.json', local_dir='./models')
hf_hub_download(repo_id='FoivosPar/Arc2Face', filename='arc2face/diffusion_pytorch_model.safetensors', local_dir='./models')
hf_hub_download(repo_id='FoivosPar/Arc2Face', filename='encoder/config.json', local_dir='./models')
hf_hub_download(repo_id='FoivosPar/Arc2Face', filename='encoder/pytorch_model.bin', local_dir='./models')
hf_hub_download(repo_id='FoivosPar/Arc2Face', filename='arcface.onnx', local_dir='./models/antelopev2')

print('Core Arc2Face models downloaded.')

Cloning into 'Arc2Face'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 168 (delta 50), reused 26 (delta 26), pack-reused 106 (from 1)
Receiving objects: 100% (168/168), 32.02 MiB | 27.14 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/Arc2Face


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

arc2face/diffusion_pytorch_model.safeten(…):   0%|          | 0.00/3.44G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

encoder/pytorch_model.bin:   0%|          | 0.00/492M [00:00<?, ?B/s]

arcface.onnx:   0%|          | 0.00/261M [00:00<?, ?B/s]

Core Arc2Face models downloaded.


## 3. Configure the antelopev2 Face-Detection Package

Arc2Face uses the `antelopev2` InsightFace bundle for face detection and ArcFace embedding
extraction. We download the bundle and remove its default `glintr100.onnx` recognition file,
because Arc2Face ships its own `arcface.onnx` (downloaded above) and having both would cause a
loading conflict.

In [ ]:
import glob

!pip install -q gdown
!gdown --id 18wEUfMNohBJ4K3Ly5wpTejPfDzp-8fI8 -O /tmp/antelopev2.zip
!unzip -q /tmp/antelopev2.zip -d ./models/

# Remove the default recognition model to avoid a conflict with Arc2Face's arcface.onnx
for f in glob.glob('./models/antelopev2/glintr100.onnx'):
    os.remove(f)
    print(f'Removed conflicting file: {f}')

print('antelopev2 configured.')

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=18wEUfMNohBJ4K3Ly5wpTejPfDzp-8fI8
From (redirected): https://drive.google.com/uc?id=18wEUfMNohBJ4K3Ly5wpTejPfDzp-8fI8&confirm=t&uuid=b1260205-9e9b-4534-a9b9-50a596104399
To: /tmp/antelopev2.zip
100% 361M/361M [00:03<00:00, 108MB/s]
Removed conflicting file: ./models/antelopev2/glintr100.onnx
antelopev2 configured.


## 4. Download and Extract the LFW Dataset

We replace the paper's original evaluation sets (Synth-500 and AgeDB) with **LFW (Labeled
Faces in the Wild)**, a public face-recognition benchmark. We download it from a Figshare
mirror and verify archive integrity before extracting (the original UMass mirror is often
unreliable).

In [ ]:
import subprocess
from pathlib import Path

LFW_TAR = '/tmp/lfw.tgz'
LFW_PATH = Path('/tmp/lfw')

if LFW_PATH.exists() and len(list(LFW_PATH.iterdir())) > 100:
    print(f'LFW already present: {len(list(LFW_PATH.iterdir()))} identities.')
else:
    print('Downloading LFW via Figshare mirror...')
    os.system(f'wget -c --show-progress "https://ndownloader.figshare.com/files/5976018" -O {LFW_TAR}')

    # Verify the archive is valid before extracting
    test = subprocess.run(['tar', '-tzf', LFW_TAR], capture_output=True)
    if test.returncode == 0:
        os.system(f'tar -xzf {LFW_TAR} -C /tmp/')
        print(f'LFW extracted: {len(list(LFW_PATH.iterdir()))} identities.')
    else:
        print('ERROR: downloaded archive is corrupted. Re-run this cell.')

LFW extracted: 5749 identities.


## 5. Filter and Sample the 50 Evaluation Identities

**This is the step that defines the shared evaluation set for the whole project.**

- We keep only identities with **≥ 2 images** (a light quality filter).
- We sample **50 identities** with a **fixed random seed (42)**, so the selection is
  deterministic and can be reproduced by the baseline notebooks.
- The first image of each identity becomes the reference input.

We limit to 50 identities due to the compute budget of a single Colab session; 50 identities
× 5 images = 250 samples per model, which is enough for a stable distribution of the
evaluation metrics.

In [ ]:
import random
import shutil

random.seed(42)  # deterministic sampling — do not change

LFW_PATH = Path('/tmp/lfw')
INPUT_PATH = Path('./dataset_lfw/inputs')
INPUT_PATH.mkdir(parents=True, exist_ok=True)

# Keep identities with at least 2 images
valid_ids = []
for person_dir in LFW_PATH.iterdir():
    if person_dir.is_dir():
        imgs = list(person_dir.glob('*.jpg'))
        if len(imgs) >= 2:
            valid_ids.append((person_dir.name, imgs))

print(f'Identities with >= 2 images: {len(valid_ids)}')

# Deterministically sample 50 identities
selected = random.sample(valid_ids, 50)

identity_map = {}
for name, imgs in selected:
    dst = INPUT_PATH / f'{name}.jpg'
    shutil.copy(imgs[0], dst)  # first image = reference input
    identity_map[name] = str(dst)

print(f'Prepared {len(identity_map)} evaluation identities.')
print(f'Examples: {list(identity_map.keys())[:5]}')

Identities with >= 2 images: 1680
Prepared 50 evaluation identities.
Examples: ['Intisar_Ajouri', 'Norman_Jewison', 'Nick_Nolte', 'Hassan_Wirajuda', 'Christopher_Patten']


## 6. Initialize the Arc2Face Pipeline

We load the fine-tuned CLIP text encoder and UNet into a Stable Diffusion v1.5 pipeline, set
the DPM-Solver++ scheduler for fast sampling, and prepare the InsightFace analyzer for
embedding extraction.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DPMSolverMultistepScheduler
from arc2face import CLIPTextModelWrapper, project_face_embs
from insightface.app import FaceAnalysis
from PIL import Image
import numpy as np

print('Loading the adapted CLIP text encoder and the fine-tuned UNet...')
encoder = CLIPTextModelWrapper.from_pretrained('models', subfolder='encoder', torch_dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained('models', subfolder='arc2face', torch_dtype=torch.float16)

print('Building the Stable Diffusion v1.5 pipeline...')
pipeline = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    text_encoder=encoder,
    unet=unet,
    torch_dtype=torch.float16,
    safety_checker=None
)
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline = pipeline.to('cuda')

print('Preparing the ArcFace identity extractor...')
app = FaceAnalysis(name='antelopev2', root='./', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

print('Arc2Face pipeline ready.')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Loading the adapted CLIP text encoder and the fine-tuned UNet...
Building the Stable Diffusion v1.5 pipeline...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Preparing the ArcFace identity extractor...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/arcface.onnx recognition ['None', 3, 112, 112] 127.5 127.5
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/scrfd_10g_bnkps.onnx detection [1, 3, '?', '?'] 127.5 128.0
set det-size: (640, 640)
Arc2Face pipeline ready.


## 7. Generation Loop (50 identities × 5 images)

For each identity we: (1) extract and L2-normalize the ArcFace embedding, (2) project it into
the CLIP token space via `project_face_embs`, and (3) sample 5 images. We use 25 DPM-Solver++
steps and `guidance_scale=3.0`, the settings recommended by the Arc2Face authors.

At the end we save `arc2face_processed.json` — **the identity list consumed by every other
notebook in this project.**

In [ ]:
import json
from tqdm import tqdm

OUTPUT_PATH = Path('./results/arc2face')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FAILED = []

def extract_embedding(img_path):
    """Extract and L2-normalize the ArcFace identity embedding from an image."""
    img = np.array(Image.open(img_path).convert('RGB'))[:, :, ::-1]  # RGB -> BGR
    faces = app.get(img)
    if not faces:
        return None
    # Largest face by bounding-box area (ignore background faces)
    face = sorted(faces, key=lambda x: (x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    emb = torch.tensor(face['embedding'], dtype=torch.float16)[None].cuda()
    emb = emb / torch.norm(emb, dim=1, keepdim=True)
    return emb

for name, img_path in tqdm(identity_map.items(), desc='Generating (Arc2Face)'):
    out_dir = OUTPUT_PATH / name
    out_dir.mkdir(exist_ok=True)

    id_emb = extract_embedding(img_path)
    if id_emb is None:
        FAILED.append(name)
        continue

    # Project the ArcFace vector into the CLIP token space
    id_emb_proj = project_face_embs(pipeline, id_emb)

    with torch.no_grad():
        images = pipeline(
            prompt_embeds=id_emb_proj,
            num_inference_steps=25,
            guidance_scale=3.0,
            num_images_per_prompt=5
        ).images

    for i, img in enumerate(images):
        img.save(out_dir / f'gen_{i:02d}.png')
    shutil.copy(img_path, out_dir / 'input.jpg')

print(f'\nDone. Failures: {len(FAILED)}')

# Save the shared identity list used by all other notebooks
processed = [n for n in identity_map if n not in FAILED]
with open('./results/arc2face_processed.json', 'w') as f:
    json.dump(processed, f)
print(f'Successfully processed identities: {len(processed)}')

Generating (Arc2Face):   0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Generating (Arc2Face):   2%|▏         | 1/50 [00:04<04:03,  4.97s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

Generating (Arc2Face):   4%|▍         | 2/50 [00:08<03:14,  4.06s/it]

...

  0%|          | 0/25 [00:00<?, ?it/s]

Generating (Arc2Face):  98%|█████████▊| 49/50 [02:48<00:03,  3.29s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

Generating (Arc2Face): 100%|██████████| 50/50 [02:52<00:00,  3.45s/it]


Done. Failures: 0
Successfully processed identities: 50


## 8. Back Up Results and Identity List to Google Drive

We store the generated images **and** `arc2face_processed.json` under
`MyDrive/Arc2Face/`. The baseline notebooks read from this exact folder.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/Arc2Face'
os.makedirs(DRIVE, exist_ok=True)

shutil.copytree('./results/arc2face', f'{DRIVE}/results_arc2face', dirs_exist_ok=True)
shutil.copy('./results/arc2face_processed.json', DRIVE)
print('Results synchronized to Google Drive.')

print('\nGoogle Drive content:')
for item in sorted(os.listdir(DRIVE)):
    p = Path(DRIVE) / item
    if p.is_dir():
        print(f'  {item}/ ({len(list(p.iterdir()))} items)')
    else:
        print(f'  {item}')

Mounted at /content/drive
Results synchronized to Google Drive.

Google Drive content:
  arc2face_processed.json
  results_arc2face/ (50 items)
